In [28]:
# import os
import hashlib
import requests
from xml.etree import ElementTree as ET
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain.chat_models import init_chat_model
import json

load_dotenv()

# Cloud LLMs (optional)
# LLM_MODEL = os.getenv("LLM_MODEL")
# LLM_PROVIDER = os.getenv("LLM_PROVIDER")

# if LLM_PROVIDER == "anthropic":
#     assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY missing from .env"


# Config
DB_DIR = "./chroma_db"
EMBED_MODEL = "nomic-embed-text"
LLM_MODEL, LLM_PROVIDER = "llama3.2", "ollama"

llm = init_chat_model(LLM_MODEL, model_provider=LLM_PROVIDER)

In [29]:
ACT_IDS = [
    "ukpga/1968/60",   # Theft Act 1968
    "ukpga/1971/48",   # Criminal Damage Act 1971
    "ukpga/1971/38",   # Misuse of Drugs Act 1971
    "ukpga/2006/35",   # Fraud Act 2006
    "ukpga/1990/18",   # Computer Misuse Act 1990
]

ACT_URLS = [f"https://www.legislation.gov.uk/{a}/data.xml" for a in ACT_IDS]
NS = "{http://www.legislation.gov.uk/namespaces/legislation}"

In [30]:
def load(url):
    # Fetches one Act's XML and parses it into a searchable tree
    req = requests.get(url)
    if req.status_code != 200:
        return None
    return ET.fromstring(req.content)


def text_of(para):
    # Pulls out all the words, skipping the (a)/(b) numbering so it doesn't glue into the sentence
    parts = []
    for node in para.iter():
        if node.tag == f"{NS}Pnumber":
            continue
        if node.text and node.text.strip():
            parts.append(node.text.strip())
        if node.tail and node.tail.strip():
            parts.append(node.tail.strip())
    return " ".join(parts)


def split(root, url):
    # Walks the Act and makes one chunk per subsection, each labelled with where it came from
    docs = []

    for p1 in root.iter(f"{NS}P1"):
        section = "".join(p1.find(f"{NS}Pnumber").itertext()).strip()
        p1_uri = p1.get("DocumentURI")

        # Schedules number their parts as paragraphs, not sections
        if p1_uri is not None and "/schedule/" in p1_uri:
            label = "Paragraph"
        else:
            label = "Section"

        for p2 in p1.iter(f"{NS}P2"):
            # No URI means this is text quoted from a different Act, so skip it
            if p2.get("DocumentURI") is None:
                continue

            sub = "".join(p2.find(f"{NS}Pnumber").itertext()).strip()
            content = text_of(p2.find(f"{NS}P2para"))

            # Built from the parent because some child URIs in the published XML are wrong
            uri = f"{p1_uri}/{sub}"

            doc = Document(
                page_content=f"{label} {section}({sub}): {content}",
                metadata={
                    "source": url,
                    "section": section,
                    "subsection": sub,
                    "uri": uri,
                    "type": label.lower(),
                },
            )
            docs.append(doc)

    return docs

In [31]:
def chunk_id(doc):
    # each provision has the same ID every run, so re-indexing overwrites instead of adding copies
    return hashlib.md5(doc.metadata["uri"].encode()).hexdigest()

def get_store():
    # retrives local vector store
    return Chroma(
        collection_name="legislation",
        embedding_function=OllamaEmbeddings(model=EMBED_MODEL),
        persist_directory=DB_DIR,
    )

In [32]:
def index(urls=ACT_URLS, rebuild=False):  # rebuild set to true -> rebuilds vector store each time
    # Fetches every Act, turns them into chunks, embeds them, and saves them to the database
    store = get_store()
    if rebuild:
        store.delete_collection()
        store = get_store()

    docs = []
    for url in urls:
        root = load(url)
        if root is None:
                    continue
        docs += split(root, url)

    print(f"{len(docs)} subsections indexed, store holds {store._collection.count()}")
    store.add_documents(docs, ids=[chunk_id(d) for d in docs])
    return store

In [33]:
def retrieve(store, question, k=4):
    # Finds the k chunks whose meaning sits closest to the question
    return store.similarity_search_with_score(question, k=k)


def show(hits): # -> for testing
    # Prints what came back, with its distance score, so you can check retrieval by eye
    for doc, score in hits:
        print(f"[{score:.3f}] s.{doc.metadata['section']}({doc.metadata['subsection']})")
        print("   ", doc.page_content[:200], "\n")

In [34]:
def build_prompt(question, hits):
    # Glues the retrieved sections onto the question, with instructions to answer only from them
    context = "\n\n".join(d.page_content for d, _ in hits)
    return (f"Answer using only the context below. Cite the section you used. "
            f"If the context does not contain the answer, say so.\n\n"
            f"Context:\n{context}\n\nQuestion: {question}")


def ask(store, question, k=4, debug=False):
    # Finds the closest sections, then asks the model to answer using only those
    hits = retrieve(store, question, k)
    if debug:
        show(hits)
    return llm.invoke(build_prompt(question, hits)).text

In [ ]:
def evaluate(store, k=4, path="data/test.json"):
    # Runs every test question with pass + fail metrics
    cases = json.load(open(path))
    hits_at_k = 0
    for case in cases:
        # pass evaluated on whether the start url of res matches expected
        results = retrieve(store, case["question"], k)
        found = any(d.metadata["uri"].startswith(case["expected"]) for d, _ in results)
        hits_at_k += found
        print(f"{'PASS' if found else 'FAIL'}  {case["question"]}")


568 subsections indexed, store holds 0
PASS  what counts as theft
PASS  what does dishonestly mean
PASS  what counts as property for theft
PASS  is taking someone's car without permission a crime
PASS  what is burglary
PASS  what is robbery
PASS  what happens if I break someone else's property
PASS  what counts as fraud by false representation
PASS  is hacking into a computer illegal
PASS  what is the penalty for supplying controlled drugs

hit@4: 10/10


In [ ]:
# create store + and run tests
store = index(rebuild=True)
evaluate(store)


568
